In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f'/Volumes/adb_db_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

# Guard: ADF may trigger this notebook even when the copy step landed no files
# (e.g. no new incremental files). Exit cleanly instead of failing on PATH_NOT_FOUND.
try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception as e:
    if 'FileNotFoundException' in str(e) or 'PATH_NOT_FOUND' in str(e) or 'does not exist' in str(e).lower():
        landing_files = []
    else:
        raise

if not landing_files:
    print(f'No landing data found at {landing_path}; skipping bronze load.')
    dbutils.notebook.exit(f'SKIPPED: no landing data at {landing_path}')

if source_file_format == 'csv':
    landing_df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(landing_path)
else:
    landing_df = spark.read.format('parquet').load(landing_path)

In [0]:
display(landing_df)

In [0]:
from pyspark.sql import functions as F

landing_df = landing_df.withColumn('landing_timestamp', F.lit(landing_timestamp))\
    .withColumn('insert_timestamp', F.current_timestamp())

display(landing_df)

In [0]:
landing_df.write.mode('append').saveAsTable(f'adb_db_caresync.bronze.{table_name}')

In [0]:
%sql
select * from adb_db_caresync.bronze.hospitals